The fixed RAG pipeline retrieves once and generates an answer. An adaptive pipeline can reject an unsuitable request, inspect retrieved evidence, rewrite the query, and try again. Those decisions can improve difficult cases, but they also add model calls, new failure paths, and opportunities to drift away from the user's question.

This lesson uses the [Week 7 release](https://github.com/jamwithai/production-agentic-rag-course/tree/week7.0), matching the local course folder. The objective is to specify the workflow as a bounded state machine, verify the implementation's actual contract, and determine whether adaptation improves supported answers enough to justify its cost. A convincing demonstration compares the adaptive system with the simpler baseline on the same evidence and questions.

## Where adaptation enters the workflow

The orchestration in `src/services/agents/agentic_rag.py` provides the context for the individual prompts. It registers guardrail, out-of-scope, retrieval, tool execution, grading, rewriting, and answer-generation nodes. Edges determine which operations can follow which results. The retrieval node explicitly constructs a tool call; it is not an unconstrained model deciding to invoke arbitrary tools.

The adaptation comes from model-assisted routing and rewritten retrieval queries inside a developer-defined workflow. That is a useful, limited form of agency. It does not imply autonomous planning, durable memory, unrestricted tool use, or that every additional decision improves the result.

```{mermaid}
flowchart TD
    G[Validate scope] -->|Accepted| R[Initiate retrieval]
    G -->|Rejected| O[Explain scope and stop]
    R -->|Budget remains| T[Run retrieval tool]
    R -->|Budget exhausted| F[Report insufficient evidence]
    T --> D[Grade retrieved evidence]
    D -->|Sufficient| A[Generate supported answer]
    D -->|Insufficient| W[Rewrite retrieval query]
    W --> R
```

This diagram makes a desired termination property explicit. A LangGraph recursion limit can provide an additional technical safeguard, but it should not substitute for an application-level retrieval and cost budget. The service should return an intelligible outcome before an internal graph limit is hit.

## State is a contract between nodes

The source's `AgentState` includes messages, original and rewritten query fields, retrieval attempts, guardrail results, source artifacts, grading results, and metadata. Its messages use the `add_messages` reducer. A node returning messages therefore interacts with a merge policy rather than simply replacing the entire transcript. Message IDs and types matter when results are updated or appended.

The original request intent and the query used for retrieval have different roles in the state. A rewrite may expand “ML” into “machine learning,” but it must not silently turn “limitations of retrieval” into “benefits of retrieval.” The original question remains the reference for answer generation and evaluation, while each retrieval query is a derived artifact with its own provenance.

The [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) formalizes state, nodes, edges, and reducers. Persistence is a separate capability supplied through checkpointers and stores. The source compiles its graph without a checkpointer, so graph state within one invocation does not establish durable conversational memory or restart recovery. Those properties depend on the application's actual persistence configuration.

Source identities should remain structured as they move from tool output into the final response. Stringifying documents and later trying to rediscover source IDs is fragile. A structured record can associate selected evidence with its tool-call ID, source/version, and retrieval query. That record supports both citation validation and an explanation of why a second retrieval changed the answer.

## Budgets, termination, and explicit outcomes

At most two retrieval attempts does not mean at most two model calls. Guardrail evaluation, document grading, rewriting, and generation each may invoke a model. Tool retries can add further work. Retrieval attempts, total calls, elapsed time, and context size therefore impose separate budget constraints.

**Experiment: verify termination without relying on an LLM.** The deterministic fixtures below let the control flow be tested independently. The grade is stipulated evidence availability, not a claim that the program understands the documents. 

1. Trace the state transitions for scope rejection and immediate evidence.
2. Repeat the trace for successful rewriting and budget exhaustion.
3. Run the fixtures and compare their paths with those predictions.


In [ ]:
def adaptive_lookup(original_query, in_scope, evidence_by_query, rewrites, limit=2):
    if limit < 1:
        raise ValueError("At least one attempt is required")
    if not in_scope:
        return {"status": "out_of_scope", "attempts": 0, "path": ["guardrail", "stop"]}
    current_query = original_query
    path = ["guardrail"]
    for attempt in range(1, limit + 1):
        path.extend(["retrieve", "grade"])
        if evidence_by_query.get(current_query, False):
            path.append("generate")
            return {"status": "supported", "attempts": attempt, "path": path,
                    "original_query": original_query, "retrieval_query": current_query}
        if attempt < limit:
            path.append("rewrite")
            current_query = rewrites.get(current_query, current_query)
    path.append("abstain")
    return {"status": "insufficient_evidence", "attempts": limit, "path": path}

cases = {
    "scope": adaptive_lookup("dog care", False, {}, {}),
    "direct": adaptive_lookup("retrieval", True, {"retrieval": True}, {}),
    "rewrite": adaptive_lookup("ML evidence", True, {"machine learning evidence": True},
                               {"ML evidence": "machine learning evidence"}),
    "exhausted": adaptive_lookup("missing fact", True, {}, {}),
}
for name, result in cases.items():
    print(name, result)
assert cases["scope"]["attempts"] == 0
assert cases["direct"]["attempts"] == 1
assert cases["rewrite"]["attempts"] == 2
assert cases["exhausted"]["status"] == "insufficient_evidence"
assert "generate" not in cases["exhausted"]["path"]


Reference output from isolated Python execution:

```text
scope {'status': 'out_of_scope', 'attempts': 0, 'path': ['guardrail', 'stop']}
direct {'status': 'supported', 'attempts': 1, 'path': ['guardrail', 'retrieve', 'grade', 'generate'], 'original_query': 'retrieval', 'retrieval_query': 'retrieval'}
rewrite {'status': 'supported', 'attempts': 2, 'path': ['guardrail', 'retrieve', 'grade', 'rewrite', 'retrieve', 'grade', 'generate'], 'original_query': 'ML evidence', 'retrieval_query': 'machine learning evidence'}
exhausted {'status': 'insufficient_evidence', 'attempts': 2, 'path': ['guardrail', 'retrieve', 'grade', 'rewrite', 'retrieve', 'grade', 'abstain']}
```

The last assertion is a behavioral contract: exhaustion cannot fall through to unsupported generation. The model separates orchestration from content quality, so a failed control-flow test cannot be blamed on a stochastic response. It does not test the LangGraph implementation itself; the source's node tests and full-stack scenarios must establish that correspondence.

The source increments attempts when preparing a retrieval tool call and returns a fallback message once the configured maximum is reached. A tool exception, an empty result, and a relevant result are distinct cases for checking whether the accounting remains consistent. A retry should not reset the counter by accidentally rebuilding the state.

## A model score is a decision signal, not a probability

The guardrail returns a score on a requested scale and compares it with a threshold, defaulting to 60 in `GraphConfig`. A score of 80 is not an 80% probability that the query is suitable. Calibration requires a labeled set and empirical agreement between scores and outcomes.

Ambiguous and failed validation require an explicit routing policy. A domain classifier should not be confused with a security boundary: an in-domain research question can still contain malicious instructions, and an out-of-domain question is not necessarily harmful. A classifier should also not decide document authorization. Permission constraints belong at the retrieval and tool boundaries, independently of the classifier's judgment.

Failure paths can change the meaning of an otherwise sensible prompt. In the local source, a missing guardrail result defaults to continuing, while a guardrail model exception creates a score of 50, which falls below the default threshold. These policies differ. Changing the threshold can change fallback behavior without changing the exception handler. An explicit degraded outcome avoids making failure handling depend on that accidental numeric relationship.

## Evidence sufficiency and the limits of length heuristics

The grading node applies a structured model output to the combined retrieved context. It records one result for `retrieved_docs`, rather than independently validating every chunk and every answer claim. That can be a reasonable initial design, but its label should not imply finer-grained assurance than it provides.

Its exception fallback considers context relevant if its stripped length exceeds 50 characters. That tests length, not relevance or sufficiency. A long paragraph about an unrelated subject passes. A compact but decisive formula could fail. This is a concrete example of **failing open** on evidence quality after a dependency error.

**Experiment: construct the smallest useful counterexample.** No language model is needed to show why the fallback cannot justify a relevance claim.


In [ ]:
question = "How does reciprocal rank fusion combine retrieval results?"
unrelated = "A detailed description of garden irrigation schedules and seasonal tomato pruning."
fallback_relevant = len(unrelated.strip()) > 50
assert fallback_relevant is True
assert "rank" not in unrelated.lower() and "retrieval" not in unrelated.lower()
print("Question:", question)
print("Unrelated context passes the length fallback:", fallback_relevant)


Reference output from isolated Python execution:

```text
Question: How does reciprocal rank fusion combine retrieval results?
Unrelated context passes the length fallback: True
```

The test disproves the interpretation “passing this fallback establishes relevance.” It does not measure the model grader's accuracy. A safer policy for evidence-sensitive answers is an explicit `grading_unavailable` outcome, with a bounded retry or an abstention. If the product permits a degraded answer, expose that state and evaluate it separately.

Rewriting has its own failure modes. Repeated expansions can add fashionable but unrelated terminology, change a negation, or retrieve the same passages again. Changes in the candidate set, evidence coverage, and query intent reveal whether a rewrite helped. A retry that introduces no new information consumes budget without creating a new opportunity to answer.

## Requested settings and effective computation

The local `src/routers/agentic_ask.py` accepts an `AskRequest`, but calls `agentic_rag.ask(query=request.query)` without forwarding the request's model, `top_k`, hybrid flag, or categories. The response then uses request fields to populate `chunks_used` and `search_mode`. This can describe requested settings rather than effective execution.

A request that changes `top_k` or model offers a direct test of this contract defect: the service invocation and trace should reveal whether the effective settings change. Echoing different parameters in the response does not establish that different models were used. The factory and runtime configuration determine the effective values.

The retriever tool also requests an embedding before calling search even when configured for lexical mode. That dependency is significant because a lexical request should not need a working embedding service unless the contract explicitly says otherwise. These cases show why tests need to cross the HTTP-to-service boundary rather than only validate a response schema.

Decision summaries and route logs are useful observability artifacts. They are not guaranteed transcripts of a model's internal reasoning, and a plausible explanation is not evidence that a result is correct. Prefer verifiable facts—queries issued, documents returned, grades recorded, attempts consumed, and outcome—to treating generated explanations as a correctness certificate.

## Docker lab: compare fixed and adaptive RAG

1. Use the local Week 7 source or a matching `week7.0` checkout, project name `rag-week7`, and configure its environment. Start the **entire** stack. The Week 7 Langfuse v3 deployment includes supporting databases, Redis, object storage, and ClickHouse; a running API alone is not the full observability deployment. Validate all declared services and inspect resource use before increasing ingestion or concurrency.

    ```bash
    docker compose --project-name rag-week7 config --services
    docker compose --project-name rag-week7 up --build -d
    docker compose --project-name rag-week7 ps
    docker compose --project-name rag-week7 exec ollama ollama pull llama3.2:1b
    docker compose --project-name rag-week7 exec airflow \
      airflow dags trigger arxiv_paper_ingestion
    curl --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:8000/api/v1/ask-agentic \
      -d '{"query":"What is a dog?","top_k":3,"use_hybrid":true,"model":"llama3.2:1b"}'
    curl --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:8000/api/v1/ask-agentic \
      -d '{"query":"How do the indexed papers use retrieval?","top_k":3,"use_hybrid":true,"model":"llama3.2:1b"}'
    docker stats --no-stream
    ```

2. The first query is a scope-rejection scenario, not a guaranteed deterministic outcome. Inspect whether it used zero retrieval attempts and produced an appropriate explanation. The second must be matched to known evidence in your corpus. Add a vague but in-scope query requiring a useful rewrite, a question with no supporting evidence, and a request that triggers an unavailable dependency.

3. Compare `/api/v1/ask` and `/api/v1/ask-agentic` on the same fixed corpus and query set. Match effective model and retrieval settings through traces, not just request bodies. Control caching so a basic endpoint's cache hit is not compared against an uncached agent run. Keep repeated trials for model variability and record answer support, correct abstention, retrieval attempts, model calls, time to first output where applicable, total latency, and failures.

4. For a rewrite scenario, verify that a second retrieval occurred and introduced useful evidence. More attempts without better support is extra cost, not adaptation success. For exhaustion, assert a bounded number of attempts and an explicit insufficient-evidence outcome. For model or grader failure, inspect whether the fallback still allows generation and whether the response discloses the degraded mode.

5. Enable Telegram only after the HTTP scenarios work, using a test bot and the release's `TELEGRAM__ENABLED` and token configuration. Confirm these names in `TelegramSettings` and `.env.example` before editing. The bot is started from the application lifecycle, rather than being a separate service in this Compose file. Multiple API worker processes can therefore create competing bot pollers; inspect worker count and ensure only one polling owner before enabling it. Compare a Telegram answer's sources and failure behavior with the same HTTP request. Keep tokens out of logs and notebooks.

## An engineering capstone: decide whether to keep the agent

1. Adaptation should earn its complexity. Define a decision rule before evaluating it: for example, improved supported-answer coverage on difficult questions without exceeding a chosen p95 latency or call budget. Choose your thresholds from the product requirement, not from whichever run looks best.

2. Perform an **ablation** by disabling one adaptive component at a time: query rewriting, grading, or domain validation. This identifies which component contributes useful behavior. Keep the corpus and held-out questions fixed. For a small labeled set, show paired per-question outcomes; an average with no uncertainty or error examples is too weak to justify a broad superiority claim.

3. Submit an engineering record with one successful recovery, one failed or unnecessary rewrite, one abstention, and one injected dependency failure. Include source and model versions, retrieved evidence, actual configuration, trace IDs, and the claim-support judgment. Propose a repair for one observed defect and write a regression test at the boundary where it occurs.

The final system is a proving ground for habits that transfer beyond RAG: explicit contracts, stable identity, reproducible representations, failure-aware orchestration, and measurement tied to user needs. A more complicated pipeline is valuable only when those habits make its behavior understandable and its benefits demonstrable.

## Sources and scope

- [Week 7 source](https://github.com/jamwithai/production-agentic-rag-course/tree/week7.0): workflow, state, tools, node fallbacks, factory, HTTP router, and Telegram lifecycle.
- [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) and [persistence](https://docs.langchain.com/oss/python/langgraph/persistence): state updates, routing, and the separate role of checkpointing.
- [OWASP prompt injection](https://genai.owasp.org/llmrisk/llm01-prompt-injection/): why domain validation and retrieved content do not replace permission enforcement.

The deterministic state-machine model and counterexample are independently authored exercises. They establish termination and fallback limitations in small cases. They do not measure a language model's grading quality, verify the LangGraph runtime, or demonstrate that adaptive RAG outperforms the baseline; the full Docker comparison is required for those conclusions.
